# Paper vs Backtest — Phase 2.6 Tool 1

Two-layer paper-vs-backtest comparison for the live trader:

**Layer 1 — signal stream (PRIMARY).** Loads `signal_events` rows for the
paper run, re-runs the same strategy in `BacktestEngine` over the matched
window, and diffs per-bar gate decisions. Answers *"does the strategy
generate the same signals in paper as in backtest, bar-for-bar?"* —
headline metric is **direction match rate**.

**Layer 2 — position stream (SECONDARY).** Once signal-stream alignment
is high, layer 2 measures the fill-side gap (entry time, entry price,
realized PnL) on positions that both sides took. Answers *"given the
same signals, how much does sandbox-vs-backtest matching engine cost
us in dollars?"*

This separation was forced by the 2026-05-30 mini-run finding: position-
level comparison alone produced direction-match = 0% on real data
because tiny EMA warmup differences flipped every marginal cross, and
because backtest fills at bar OPEN vs paper fills at next-tick-after-CLOSE
(a 15-min systematic offset). Both issues are visible at the signal layer
but invisible at the position layer.

**Modes:**
- `USE_SYNTHETIC_DATA = True` (default) — runs end-to-end on a synthetic
  bar series, with the "paper" stream perturbed to inject the canonical
  Phase 2.6 finding shapes (1-bar lag + 20bps slippage).
- `USE_SYNTHETIC_DATA = False` — loads from PG and re-runs the backtest
  over the matched window. Set `RUN_ID` and `INSTRUMENT_ID` in cell 2.

In [ ]:
# Cell 1: Imports + config

from datetime import UTC, datetime
from decimal import Decimal

import asyncpg
import matplotlib.pyplot as plt
import pandas as pd

from src.config.settings import get_settings
from src.core import bar_type_str, get_venue_config
from utils import (
    compare_position_streams,
    compute_backtest_warmup_start,
    join_signal_streams,
    load_backtest_data,
    load_paper_positions,
    load_signal_events,
    run_backtest_position_stream,
    run_backtest_signal_stream,
    summarize_signal_streams,
)

settings = get_settings()
CATALOG_PATH = '../data/catalog'
print(f'DB: {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}')

In [ ]:
# Cell 2: Mode toggle + run identification

USE_SYNTHETIC_DATA = True   # flip to False once a clean Stage A/B run_id exists

# REAL mode: set these from a row in strategy_runs.
RUN_ID         = ''                                  # UUID string
INSTRUMENT_ID  = 'BTC-USD-PERP.HYPERLIQUID'
BAR_INTERVAL   = '4h'

# Strategy hyperparameters — MUST match what paper actually ran with.
P26_FAST = 10
P26_SLOW = 40
P26_MA_TYPE = 'EMA'

# Layer 2 position-pair match tolerance.
MATCH_TOL_BARS = 1.0
BAR_SECONDS = {'1m': 60, '5m': 300, '15m': 900, '1h': 3600, '4h': 14400, '1d': 86400}[BAR_INTERVAL]

# Backtest warmup multiplier — how many slow_period windows of extra
# data to load before paper's first signal so the backtest's EMAs
# approximately match paper's at the start of the comparison window.
# Default 2.0 matches NT's typical RequestBars on startup. Tune up if
# direction-match rate is unexpectedly low.
WARMUP_MULTIPLIER = 2.0

print(f'mode: {"SYNTHETIC" if USE_SYNTHETIC_DATA else "REAL"}')
print(f'params: fast={P26_FAST} slow={P26_SLOW} ma_type={P26_MA_TYPE}')
print(f'match tol: {MATCH_TOL_BARS} bars ({MATCH_TOL_BARS * BAR_SECONDS}s)')
print(f'warmup multiplier: {WARMUP_MULTIPLIER}x slow_period bars')

In [ ]:
# Cell 3: Build paper + backtest signal + position streams

if USE_SYNTHETIC_DATA:
    # Synthetic mode — produce both signal and position streams from
    # a single backtest run, then perturb the paper position stream.
    # Signal streams are identical between paper and backtest in this
    # mode by construction.
    from nautilus_trader.model.data import Bar, BarType
    from nautilus_trader.model.objects import Price, Quantity
    from nautilus_trader.test_kit.providers import TestInstrumentProvider

    _inst = TestInstrumentProvider.btcusdt_perp_binance()
    _bt_str = bar_type_str(str(_inst.id), '1d')
    _bar_type = BarType.from_str(_bt_str)
    _closes = (
        [100.0 - i for i in range(30)]
        + [70.0 + i for i in range(60)]
        + [130.0 - i for i in range(30)]
    )
    _base = int(datetime(2024, 1, 1, tzinfo=UTC).timestamp() * 1e9)
    _one_day = 86_400 * 10**9
    _px_fmt = f'{{:.{int(_inst.price_precision)}f}}'
    _qty_fmt = f'{{:.{int(_inst.size_precision)}f}}'
    _bars = [
        Bar(_bar_type,
            Price.from_str(_px_fmt.format(c-0.5)),
            Price.from_str(_px_fmt.format(c+1.0)),
            Price.from_str(_px_fmt.format(c-1.0)),
            Price.from_str(_px_fmt.format(c)),
            Quantity.from_str(_qty_fmt.format(1.0)),
            _base + i * _one_day, _base + i * _one_day)
        for i, c in enumerate(_closes)
    ]
    _vc = get_venue_config('HYPERLIQUID_PERP')
    paper_signals = backtest_signals = run_backtest_signal_stream(
        instrument=_inst, bars=_bars, venue_config=_vc,
        fast_period=3, slow_period=5, ma_type='EMA', leverage=1,
    )
    backtest_positions = run_backtest_position_stream(
        instrument=_inst, bars=_bars, venue_config=_vc,
        fast_period=3, slow_period=5, ma_type='EMA', leverage=1,
    )
    paper_positions = backtest_positions.copy()
    paper_positions['ts_opened'] = paper_positions['ts_opened'] + pd.Timedelta(days=1)
    paper_positions['ts_closed'] = paper_positions['ts_closed'] + pd.Timedelta(days=1)
    paper_positions['avg_px_open'] = paper_positions['avg_px_open'].apply(
        lambda d: d * Decimal('1.0020')
    )
    paper_positions['realized_pnl'] = paper_positions.apply(
        lambda r: r['realized_pnl'] - Decimal('0.0020') * r['avg_px_open'] * r['peak_qty'],
        axis=1,
    )
    BAR_SECONDS = 86_400
else:
    # REAL mode — load paper streams from PG, then re-run the backtest
    # with WARMUP-MATCHED data so the EMAs are primed equivalently when
    # the comparison window begins.
    if not RUN_ID:
        raise RuntimeError('Set RUN_ID + INSTRUMENT_ID in cell 2 first.')

    paper_signals = await load_signal_events(settings.postgres_dsn, RUN_ID)
    if paper_signals.empty:
        raise RuntimeError(f'No signal_events in PG for run {RUN_ID}.')
    paper_positions = await load_paper_positions(settings.postgres_dsn, RUN_ID)

    bt_start = compute_backtest_warmup_start(
        paper_signals['ts'].min(),
        slow_period=P26_SLOW, bar_seconds=BAR_SECONDS,
        multiplier=WARMUP_MULTIPLIER,
    )
    bt_end = (
        paper_positions['ts_closed'].max() if not paper_positions.empty
        else paper_signals['ts'].max()
    )
    print(f'paper signal window: {paper_signals["ts"].min()}  ->  {paper_signals["ts"].max()}')
    print(f'backtest data window: {bt_start}  ->  {bt_end}')
    print(f'  ({WARMUP_MULTIPLIER * P26_SLOW:.0f} slow-bar warmup before paper start)')

    bar_type_string = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
    _vc = get_venue_config('HYPERLIQUID_PERP')
    instrument, bars = load_backtest_data(
        CATALOG_PATH, INSTRUMENT_ID, bar_type_string,
        venue_config=_vc, date_start=bt_start.isoformat(), date_end=bt_end.isoformat(),
    )
    backtest_signals = run_backtest_signal_stream(
        instrument=instrument, bars=bars, venue_config=_vc,
        fast_period=P26_FAST, slow_period=P26_SLOW, ma_type=P26_MA_TYPE,
    )
    backtest_positions = run_backtest_position_stream(
        instrument=instrument, bars=bars, venue_config=_vc,
        fast_period=P26_FAST, slow_period=P26_SLOW, ma_type=P26_MA_TYPE,
    )

print()
print(f'paper signals:      {len(paper_signals):>6}')
print(f'backtest signals:   {len(backtest_signals):>6}')
print(f'paper positions:    {len(paper_positions):>6}')
print(f'backtest positions: {len(backtest_positions):>6}')

In [ ]:
# Cell 4: Layer 1 — signal-stream comparison (PRIMARY metric)

joined = join_signal_streams(paper_signals, backtest_signals)
summary = summarize_signal_streams(joined)

print('Phase 2.6 Tool 1 — LAYER 1: signal-stream comparison')
print('=' * 60)
print(f'Matched bars:                {summary["matched_bars"]:>6}')
print(f'Paper-only bars:             {summary["paper_only_bars"]:>6}  (signal_events paper logged but backtest did not)')
print(f'Backtest-only bars:          {summary["bt_only_bars"]:>6}  (backtest bars before paper started)')

if summary['matched_bars'] > 0:
    pct = lambda v: f'{v*100:5.1f}%' if v is not None else '   n/a'
    print()
    print(f'Direction match rate:        {pct(summary["direction_match_rate"])}    <- PRIMARY METRIC')
    print(f'Acted-action match rate:     {pct(summary["acted_match_rate"])}')
    print()
    print(f'Paper acted (cross fired):   {summary["paper_acted_count"]:>6}')
    print(f'Backtest acted:              {summary["bt_acted_count"]:>6}')
    print(f'Acted on both sides:         {summary["acted_on_both_count"]:>6}')
    print(f'  Same direction:            {summary["acted_same_direction_count"]:>6}')
    print(f'  Opposite direction:        {summary["acted_opposite_direction_count"]:>6}    <- worst-case divergence')

    if summary['direction_match_rate'] is not None and summary['direction_match_rate'] < 0.9:
        print()
        print('WARNING: direction match rate < 90%. Position-layer gap numbers')
        print('below are NOT meaningful until signal-layer alignment improves.')
        print('Common causes:')
        print('  - EMA warmup mismatch: increase WARMUP_MULTIPLIER in cell 2')
        print('  - Paper trader used different MA params than P26_FAST/SLOW')
        print('  - Sandbox bar feed vs catalog bar feed have OHLCV drift')

joined.head(10)

In [ ]:
# Cell 5: Divergence walk (where paper and backtest disagreed)

if joined.empty:
    print('No matched bars — nothing to plot.')
else:
    divergent = joined[joined['divergent']]
    print(f'{len(divergent)} divergent bars out of {len(joined)} matched '
          f'({len(divergent)/len(joined)*100:.1f}%)')

    if not divergent.empty:
        acted_divergent = divergent[divergent['paper_acted'] | divergent['bt_acted']]
        print(f'  ...of which {len(acted_divergent)} had at least one side ACT on the divergent signal')
        print()
        print('First 10 divergent bars where at least one side acted:')
        display(
            acted_divergent[['ts', 'paper_signal', 'paper_acted', 'bt_signal', 'bt_acted',
                             'paper_fast', 'paper_slow', 'bt_fast', 'bt_slow']]
            .head(10)
        )

        fig, ax = plt.subplots(1, 1, figsize=(15, 4))
        ax.scatter(joined['ts'], joined['paper_signal'], label='paper',
                   marker='o', s=30, alpha=0.6, color='#4C9AFF')
        ax.scatter(joined['ts'], joined['bt_signal'], label='backtest',
                   marker='x', s=30, alpha=0.6, color='#FF8B00')
        ax.set_title('Signal direction per bar — divergences are where the two markers vertically separate')
        ax.set_ylabel('signal (+1 / -1)')
        ax.set_yticks([-1, 0, 1])
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# Cell 6: Layer 2 — position-stream comparison (SECONDARY, gated on signal alignment)

# Layer 2 only produces interpretable numbers when layer 1's direction-
# match rate is high (rule of thumb: >=90%). If signals diverge, paired
# positions will be opposite-direction trades that don't actually
# correspond to each other and PnL gaps are meaningless.

SIGNAL_ALIGNMENT_THRESHOLD = 0.90
_signal_ok = (
    summary['matched_bars'] > 0
    and summary['direction_match_rate'] is not None
    and summary['direction_match_rate'] >= SIGNAL_ALIGNMENT_THRESHOLD
)

if not _signal_ok:
    print('Skipping layer 2 — signal alignment below threshold '
          f'({SIGNAL_ALIGNMENT_THRESHOLD*100:.0f}%). Fix the signal layer first.')
    matched_pos = pd.DataFrame()
else:
    matched_pos = compare_position_streams(
        paper_positions, backtest_positions,
        tol=MATCH_TOL_BARS, tol_unit='bars', bar_seconds=BAR_SECONDS,
    )

    n_matched = len(matched_pos)
    n_paper_unmatched = matched_pos.attrs['paper_unmatched']
    n_bt_unmatched = matched_pos.attrs['bt_unmatched']

    print('Phase 2.6 Tool 1 — LAYER 2: position-stream gap')
    print('=' * 60)
    print(f'Matched position pairs:      {n_matched}')
    print(f'Paper-only positions:        {n_paper_unmatched}')
    print(f'Backtest-only positions:     {n_bt_unmatched}')

    if not matched_pos.empty:
        print()
        print('Gap distributions (paper - backtest):')
        print(f'  entry_time_gap_s :  median={matched_pos["entry_time_gap_s"].median():>10.1f}s  '
              f'mean={matched_pos["entry_time_gap_s"].mean():>10.1f}s')
        print(f'  entry_px_gap_bps :  median={matched_pos["entry_px_gap_bps"].median():>10.3f}   '
              f'mean={matched_pos["entry_px_gap_bps"].mean():>10.3f}')
        print(f'  pnl_gap (USD)    :  median={matched_pos["pnl_gap"].median():>10.3f}    '
              f'sum={matched_pos["pnl_gap"].sum():>10.3f}')

matched_pos.head(10) if not matched_pos.empty else None

In [ ]:
# Cell 7: Layer 2 — gap histograms (only when signal alignment passed)

if matched_pos.empty:
    print('Skipped (signal alignment too low or no matched positions).')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ax1, ax2, ax3 = axes

    ax1.hist(matched_pos['entry_time_gap_s'] / 60, bins=30, color='#4C9AFF', edgecolor='white')
    ax1.axvline(0, color='red', linestyle='--', linewidth=1)
    ax1.set_title(f'Entry-time gap (paper - bt), minutes  |  median={matched_pos["entry_time_gap_s"].median()/60:.1f}')
    ax1.set_xlabel('minutes')

    ax2.hist(matched_pos['entry_px_gap_bps'], bins=30, color='#FF8B00', edgecolor='white')
    ax2.axvline(0, color='red', linestyle='--', linewidth=1)
    ax2.set_title(f'Entry-price gap, bps  |  median={matched_pos["entry_px_gap_bps"].median():.2f}')
    ax2.set_xlabel('bps')

    ax3.hist(matched_pos['pnl_gap'], bins=30, color='#36B37E', edgecolor='white')
    ax3.axvline(0, color='red', linestyle='--', linewidth=1)
    ax3.set_title(f'Realized PnL gap (USD)  |  total={matched_pos["pnl_gap"].sum():.2f}')
    ax3.set_xlabel('USD')

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 8: Verdict snapshot — paste into reports/decisions/ (gitignored)

if joined.empty:
    print('No matched bars — no verdict to write.')
else:
    verdict_lines = [
        '## Phase 2.6 Tool 1 — paper-vs-backtest verdict',
        '',
        '### Layer 1: signal-stream alignment (PRIMARY)',
        '',
        '| Metric | Value |',
        '|---|---|',
        f'| Matched bars | {summary["matched_bars"]} |',
        f'| Paper-only bars | {summary["paper_only_bars"]} |',
        f'| Backtest-only bars | {summary["bt_only_bars"]} |',
        f'| **Direction match rate** | **{(summary["direction_match_rate"] or 0)*100:.1f}%** |',
        f'| Acted-action match rate | {(summary["acted_match_rate"] or 0)*100:.1f}% |',
        f'| Acted on both sides | {summary["acted_on_both_count"]} |',
        f'| Acted SAME direction | {summary["acted_same_direction_count"]} |',
        f'| Acted OPPOSITE direction | {summary["acted_opposite_direction_count"]} |',
        '',
    ]
    if _signal_ok and not matched_pos.empty:
        paper_total = matched_pos['paper_pnl'].astype(float).sum()
        bt_total = matched_pos['bt_pnl'].astype(float).sum()
        haircut_pct = (
            (paper_total - bt_total) / abs(bt_total) * 100 if bt_total else 0.0
        )
        verdict_lines += [
            '### Layer 2: position-stream gap (SECONDARY)',
            '',
            '| Metric | Value |',
            '|---|---|',
            f'| Matched position pairs | {len(matched_pos)} |',
            f'| Entry-time gap median | {matched_pos["entry_time_gap_s"].median():.0f}s |',
            f'| Entry-time gap p95 | {matched_pos["entry_time_gap_s"].abs().quantile(0.95):.0f}s |',
            f'| Entry-price gap median | {matched_pos["entry_px_gap_bps"].median():.2f} bps |',
            f'| Entry-price gap p95 | {matched_pos["entry_px_gap_bps"].abs().quantile(0.95):.2f} bps |',
            f'| PnL gap total | {matched_pos["pnl_gap"].sum():.2f} USD |',
            f'| Realized PnL (paper / backtest) | {paper_total:.2f} / {bt_total:.2f} USD |',
            f'| **Haircut** | **{haircut_pct:+.1f}%** |',
            '',
        ]
    else:
        verdict_lines += [
            '### Layer 2: position-stream gap',
            '',
            'SKIPPED — signal-layer direction match below threshold. Fix signal',
            'alignment before measuring fill-side gaps.',
            '',
        ]

    print('\n'.join(verdict_lines))